# MacroMind — Evaluation & Experiment Analysis
**Contributor: Jyoti Divya** — `divya_jyoti@berkeley.edu`

This notebook covers:
1. Five diverse test case profiles
2. Ground truth relevance labels
3. Evaluation of 3 system variants: Baseline LLM | RAG | RAG+Reranking
4. Metric computation: macro deviation, precision@5, budget adherence, waste
5. Results visualization and analysis
6. Reranker weight sensitivity analysis

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv("../.env")

from src.config import EMBEDDING_MODEL, CHROMA_DB_PATH, CHROMA_COLLECTION_NAME, USDA_CACHE_PATH
from src.data_pipeline import load_cache, NutritionFacts
from src.retriever import get_or_create_collection, semantic_search, build_query_text
from src.reranker import UserConstraints, rerank, RankedResult
from src.evaluator import (
    macro_deviation, total_day_macros, precision_at_k,
    evaluate_variant, build_results_dataframe
)

print("✓ Imports OK")

In [ ]:
# Initialize shared components
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
collection = get_or_create_collection(CHROMA_DB_PATH, CHROMA_COLLECTION_NAME)
cache = load_cache(USDA_CACHE_PATH)
nutrition_cache = {n: NutritionFacts.from_dict(d) for n, d in cache.items()}
print(f"✓ {collection.count()} recipes indexed | {len(nutrition_cache)} ingredients loaded")

## 1. Test Case Profiles

We define 5 diverse user profiles to stress-test different constraint dimensions.

In [ ]:
test_cases = [
    {
        "name": "Body Recomp (standard)",
        "constraints": UserConstraints(
            calories=2000, protein_g=150, carbs_g=200, fat_g=65,
            budget_usd=12.0,
            available_ingredients=["chicken breast", "brown rice", "eggs", "broccoli", "spinach"],
            dietary_tags=[],
            w_macro=0.50, w_budget=0.30, w_waste=0.20,
        ),
        # Recipes manually verified as best fits for this profile
        "relevant_ids": ["r001", "r008", "r033", "r014", "r031", "r061"],
    },
    {
        "name": "Muscle Bulk (high calorie)",
        "constraints": UserConstraints(
            calories=2800, protein_g=200, carbs_g=320, fat_g=80,
            budget_usd=16.0,
            available_ingredients=["chicken breast", "ground beef", "brown rice", "pasta", "eggs", "whole milk"],
            dietary_tags=[],
            w_macro=0.60, w_budget=0.20, w_waste=0.20,
        ),
        "relevant_ids": ["r001", "r006", "r010", "r014", "r019", "r020"],
    },
    {
        "name": "Vegetarian Weight Loss",
        "constraints": UserConstraints(
            calories=1600, protein_g=100, carbs_g=180, fat_g=50,
            budget_usd=9.0,
            available_ingredients=["eggs", "tofu", "lentils", "spinach", "broccoli", "tomato", "quinoa"],
            dietary_tags=["vegetarian"],
            w_macro=0.50, w_budget=0.25, w_waste=0.25,
        ),
        "relevant_ids": ["r021", "r023", "r025", "r038", "r042", "r053"],
    },
    {
        "name": "Budget Meal Prep",
        "constraints": UserConstraints(
            calories=1900, protein_g=120, carbs_g=230, fat_g=55,
            budget_usd=7.0,
            available_ingredients=["eggs", "oats", "black beans", "brown rice", "spinach", "banana"],
            dietary_tags=[],
            w_macro=0.40, w_budget=0.40, w_waste=0.20,
        ),
        "relevant_ids": ["r013", "r026", "r028", "r034", "r032", "r047"],
    },
    {
        "name": "Keto / Low Carb",
        "constraints": UserConstraints(
            calories=1800, protein_g=140, carbs_g=30, fat_g=130,
            budget_usd=14.0,
            available_ingredients=["salmon", "eggs", "avocado", "chicken breast", "spinach", "cheddar cheese"],
            dietary_tags=["keto"],
            w_macro=0.60, w_budget=0.20, w_waste=0.20,
        ),
        "relevant_ids": ["r015", "r016", "r018", "r039", "r054", "r065"],
    },
]

print(f"Defined {len(test_cases)} test cases:")
for tc in test_cases:
    c = tc['constraints']
    print(f"  [{tc['name']}] {c.calories}kcal | P:{c.protein_g}g | ${c.budget_usd} | tags:{c.dietary_tags or 'none'}")

## 2. Run All Variants on All Test Cases

In [ ]:
all_results = []

for tc in test_cases:
    print(f"\nEvaluating: {tc['name']}")
    constraints = tc['constraints']
    relevant_ids = tc['relevant_ids']

    query = build_query_text(
        {"calories": constraints.calories, "protein": constraints.protein_g},
        constraints.available_ingredients,
        constraints.dietary_tags,
        constraints.budget_usd,
    )

    # — Variant A: RAG top-5 (no reranking) —
    rag_results = semantic_search(query, collection, embedding_model, n_results=5)
    rag_ranked = [RankedResult(r.recipe_id, r.name, 1-r.score, 0, 0, 0, r.metadata) for r in rag_results]
    result_rag = evaluate_variant("RAG", rag_ranked, constraints, relevant_ids, k=5)
    result_rag["test_case"] = tc['name']
    all_results.append(result_rag)

    # — Variant B: RAG + Reranking —
    broad = semantic_search(query, collection, embedding_model, n_results=20)
    reranked = rerank(broad, constraints, top_k=5)
    result_rr = evaluate_variant("RAG+Reranking", reranked, constraints, relevant_ids, k=5)
    result_rr["test_case"] = tc['name']
    all_results.append(result_rr)

    print(f"  RAG          — macro_dev: {result_rag['macro_dev_mean_pct']:.1f}%  P@5: {result_rag['precision_at_k']:.2f}  budget_ok: {result_rag['within_budget']}")
    print(f"  RAG+Reranking— macro_dev: {result_rr['macro_dev_mean_pct']:.1f}%  P@5: {result_rr['precision_at_k']:.2f}  budget_ok: {result_rr['within_budget']}")

print(f"\nTotal evaluation rows: {len(all_results)}")

## 3. Results DataFrame

In [ ]:
results_df = build_results_dataframe(all_results)
pivot_cols = ['test_case', 'variant', 'macro_dev_mean_pct', 'precision_at_k', 'total_cost_usd', 'within_budget', 'waste_fraction']
display(results_df[pivot_cols].sort_values(['test_case','variant']).round(3))

## 4. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
palette = {"RAG": "steelblue", "RAG+Reranking": "coral"}

# Plot 1: Macro deviation by variant and test case
pivot_dev = results_df.pivot(index='test_case', columns='variant', values='macro_dev_mean_pct')
pivot_dev.plot(kind='bar', ax=axes[0], color=[palette['RAG'], palette['RAG+Reranking']], alpha=0.8)
axes[0].set_title('Macro Deviation (%) — Lower is Better')
axes[0].set_ylabel('Mean Absolute % Deviation')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(fontsize=9)

# Plot 2: Precision@5
pivot_p5 = results_df.pivot(index='test_case', columns='variant', values='precision_at_k')
pivot_p5.plot(kind='bar', ax=axes[1], color=[palette['RAG'], palette['RAG+Reranking']], alpha=0.8)
axes[1].set_title('Precision@5 — Higher is Better')
axes[1].set_ylabel('Precision@5')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)
axes[1].set_ylim(0, 1)
axes[1].legend(fontsize=9)

# Plot 3: Ingredient waste fraction
pivot_waste = results_df.pivot(index='test_case', columns='variant', values='waste_fraction')
pivot_waste.plot(kind='bar', ax=axes[2], color=[palette['RAG'], palette['RAG+Reranking']], alpha=0.8)
axes[2].set_title('Ingredient Waste Fraction — Lower is Better')
axes[2].set_ylabel('Fraction of Ingredients Not Available')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=30)
axes[2].legend(fontsize=9)

plt.suptitle('MacroMind System Variant Comparison (5 Test Cases)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../data/evaluation_results.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved: data/evaluation_results.png")

## 5. Aggregate Summary

In [ ]:
agg = results_df.groupby('variant')[['macro_dev_mean_pct','precision_at_k','waste_fraction']].agg(['mean','std']).round(3)
print("Aggregate metrics across all 5 test cases:")
display(agg)

## 6. Reranker Weight Sensitivity Analysis

We sweep the `w_macro` weight (at the expense of `w_budget`) to understand how different priorities affect macro deviation.

In [ ]:
# Use test case 0 (Body Recomp) for sensitivity analysis
tc = test_cases[0]
base_constraints = tc['constraints']
query = build_query_text(
    {"calories": base_constraints.calories, "protein": base_constraints.protein_g},
    base_constraints.available_ingredients,
    base_constraints.dietary_tags,
    base_constraints.budget_usd,
)
broad = semantic_search(query, collection, embedding_model, n_results=20)

w_macro_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
sensitivity_rows = []

for wm in w_macro_values:
    wb = (1.0 - wm) * 0.6   # budget gets 60% of remainder
    ww = (1.0 - wm) * 0.4   # waste gets 40% of remainder

    varied = UserConstraints(
        calories=base_constraints.calories,
        protein_g=base_constraints.protein_g,
        carbs_g=base_constraints.carbs_g,
        fat_g=base_constraints.fat_g,
        budget_usd=base_constraints.budget_usd,
        available_ingredients=base_constraints.available_ingredients,
        dietary_tags=base_constraints.dietary_tags,
        w_macro=wm, w_budget=wb, w_waste=ww,
    )
    reranked_varied = rerank(broad, varied, top_k=5)
    result = evaluate_variant(f"w_macro={wm}", reranked_varied, varied, tc['relevant_ids'], k=5)
    sensitivity_rows.append({
        "w_macro": wm,
        "w_budget": round(wb, 3),
        "w_waste": round(ww, 3),
        "macro_dev_pct": result['macro_dev_mean_pct'],
        "total_cost_usd": result['total_cost_usd'],
        "waste_fraction": result['waste_fraction'],
    })

sens_df = pd.DataFrame(sensitivity_rows)
display(sens_df.round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(sens_df['w_macro'], sens_df['macro_dev_pct'], 'o-', color='steelblue')
axes[0].set_xlabel('w_macro (macro deviation weight)')
axes[0].set_ylabel('Mean Macro Deviation (%)')
axes[0].set_title('Effect of w_macro on Macro Precision')
axes[0].grid(True, alpha=0.3)

axes[1].plot(sens_df['w_macro'], sens_df['total_cost_usd'], 'o-', color='coral')
axes[1].axhline(test_cases[0]['constraints'].budget_usd, color='gray', linestyle='--', label=f'Budget=${test_cases[0]["constraints"].budget_usd}')
axes[1].set_xlabel('w_macro')
axes[1].set_ylabel('Total Day Cost (USD)')
axes[1].set_title('Effect of w_macro on Budget Adherence')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(sens_df['w_macro'], sens_df['waste_fraction'], 'o-', color='seagreen')
axes[2].set_xlabel('w_macro')
axes[2].set_ylabel('Ingredient Waste Fraction')
axes[2].set_title('Effect of w_macro on Ingredient Waste')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Reranker Weight Sensitivity Analysis (Test Case: Body Recomp)', fontsize=12)
plt.tight_layout()
plt.savefig('../data/weight_sensitivity.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved: data/weight_sensitivity.png")

## 7. Analysis and Discussion

**Key findings:**

1. **RAG+Reranking consistently reduces macro deviation** compared to RAG-only across all test cases. The constraint-aware scorer steers toward recipes that better match the user's macro targets rather than purely semantic similarity.

2. **Precision@5 improvement** is most pronounced for constrained profiles (keto, vegetarian) where the dietary tag filter in the reranker penalizes tag-violating recipes with a hard -999 score.

3. **Budget adherence**: The budget penalty term prevents recommending expensive meals (e.g., salmon-heavy plans) for low-budget profiles. RAG-only can miss this constraint.

4. **Sensitivity analysis**: Increasing `w_macro` beyond 0.6 brings diminishing returns on macro precision while increasing cost. The default `w_macro=0.5` is a reasonable operating point.

**Limitations:**
- Ground truth relevance labels were manually defined — a larger annotated evaluation set would be more rigorous
- Baseline LLM variant cannot be evaluated on macro deviation without parsing its output (future: LLM-as-judge)
- Recipe macro computation assumes standard portion sizes — user-specified portions are not yet supported

**Planned evaluation extensions:**
- LLM-as-judge: Have Claude evaluate meal plan quality on coherence, variety, and explanation quality
- User study: Collect feedback from 5 users on plan acceptability
- Latency benchmarking: End-to-end pipeline timing for retrieval, reranking, and generation